In [ ]:
#install clip
!pip install ftfy regex tqdm
!pip install openai_clip

In [ ]:
#necessary imports
import torch
import torchvision
import torch.nn as nn
import clip
from torch.nn import functional as F
from tqdm import tqdm
from collections import OrderedDict

We will now create functions to correctly get our data and split it into base and novel classes.

In [ ]:
def get_data(data_dir="./data", transform=None):
    """Load Flowers102 train, validation and test sets.
    Args:
        data_dir (str): Directory where the dataset will be stored.
        transform (torch.Compose)
    Returns:
        tuple: A tuple containing the train, validation, and test sets.
    """
    train = torchvision.datasets.Flowers102(root=data_dir, split="train", download=True, transform=transform)
    val = torchvision.datasets.Flowers102(root=data_dir, split="val", download=True, transform=transform)
    test = torchvision.datasets.Flowers102(root=data_dir, split="test", download=True, transform=transform)
    return train, val, test

In [ ]:
def base_novel_categories(dataset):
    # set returns the unique set of all dataset classes
    all_classes = set(dataset._labels)
    # and let's count them
    num_classes = len(all_classes)

    # here list(range(num_classes)) returns a list from 0 to num_classes - 1
    # then we slice the list in half and generate base and novel category lists
    base_classes = list(range(num_classes))[:num_classes//2]
    novel_classes = list(range(num_classes))[num_classes//2:]
    return base_classes, novel_classes

In [ ]:
_, _, tmp_test = get_data()
base_classes, novel_classes = base_novel_categories(tmp_test)
CLASS_NAMES = ["pink primrose", "hard-leaved pocket orchid", "canterbury bells", "sweet pea",
                "english marigold", "tiger lily", "moon orchid", "bird of paradise", "monkshood",
                "globe thistle", "snapdragon", "colt's foot", "king protea", "spear thistle",
                "yellow iris", "globe-flower", "purple coneflower", "   ", "balloon flower",
                "giant white arum lily", "fire lily", "pincushion flower", "fritillary", "red ginger",
                "grape hyacinth", "corn poppy", "prince of wales feathers", "stemless gentian", "artichoke",
                "sweet william", "carnation", "garden phlox", "love in the mist", "mexican aster",
                "alpine sea holly", "ruby-lipped cattleya", "cape flower", "great masterwort", "siam tulip",
                "lenten rose", "barbeton daisy", "daffodil", "sword lily", "poinsettia", "bolero deep blue",
                "wallflower", "marigold", "buttercup", "oxeye daisy", "common dandelion", "petunia", "wild pansy",
                "primula", "sunflower", "pelargonium", "bishop of llandaff", "gaura", "geranium", "orange dahlia",
                "pink-yellow dahlia", "cautleya spicata", "japanese anemone", "black-eyed susan", "silverbush",
                "californian poppy", "osteospermum", "spring crocus", "bearded iris", "windflower", "tree poppy",
                "gazania", "azalea", "water lily", "rose", "thorn apple", "morning glory", "passion flower", "lotus",
                "toad lily", "anthurium", "frangipani", "clematis", "hibiscus", "columbine", "desert-rose",
                "tree mallow", "magnolia", "cyclamen", "watercress", "canna lily", "hippeastrum", "bee balm",
                "ball moss", "foxglove", "bougainvillea", "camellia", "mallow", "mexican petunia", "bromelia",
                "blanket flower", "trumpet creeper", "blackberry lily"]
print("Base Class Names:", [(i, CLASS_NAMES[i]) for i in base_classes])
print("Novel Class Names:", [(i, CLASS_NAMES[i]) for i in novel_classes])

Let's now split the dataset.

In [ ]:
def split_data(dataset, base_classes):
    # these two lists will store the sample indexes
    base_categories_samples = []
    novel_categories_samples = []

    # we create a set of base classes to compute the test below in O(1)
    # this is optional and can be removed
    base_set = set(base_classes)

    # here we iterate over sample labels and also get the correspondent sample index
    for sample_id, label in enumerate(dataset._labels):
        if label in base_set:
            base_categories_samples.append(sample_id)
        else:
            novel_categories_samples.append(sample_id)

    # here we create the dataset subsets
    # the torch Subset is just a wrapper around the dataset
    # it simply stores the subset indexes and the original dataset (your_subset.dataset)
    # when asking for sample i in the subset, torch will look for its original position in the dataset and retrieve it
    # https://pytorch.org/docs/stable/data.html#torch.utils.data.Subset
    base_dataset = torch.utils.data.Subset(dataset, base_categories_samples)
    novel_dataset = torch.utils.data.Subset(dataset, novel_categories_samples)
    return base_dataset, novel_dataset

In [ ]:
def create_remapped_dataset(dataset, selected_classes):
    """Create a dataset subset with remapped labels.

    Args:
        dataset: Original dataset
        selected_classes: List of class indices to include

    Returns:
        Subset dataset with labels remapped to [0, len(selected_classes)-1]
    """
    # Create mapping from original labels to new labels
    label_map = {old_label: new_label for new_label, old_label in enumerate(selected_classes)}
    selected_set = set(selected_classes)

    # Find samples and create new labels
    selected_samples = []
    new_labels = []

    for sample_id, label in enumerate(dataset._labels):
        if label in selected_set:
            selected_samples.append(sample_id)
            new_labels.append(label_map[label])

    # Create subset
    subset = torch.utils.data.Subset(dataset, selected_samples)

    # Add remapped labels to subset
    subset.remapped_labels = new_labels

    return subset

class RemappedDataset(torch.utils.data.Dataset):
    """Wrapper dataset that returns remapped labels"""
    def __init__(self, subset_dataset):
        self.dataset = subset_dataset.dataset
        self.indices = subset_dataset.indices
        self.labels = subset_dataset.remapped_labels

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        # Get original sample
        original_idx = self.indices[idx]
        image, _ = self.dataset[original_idx]  # Ignore original label

        # Return with remapped label
        return image, self.labels[idx]

Let's now load a pretrained CLIP model.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

# Load the CLIP model and preprocessing transform
model, preprocess = clip.load("ViT-B/16", device=device)

# Cast the model to float32 if on GPU to prevent dtype errors
if device == "cuda":
    model.float()

model.eval()  # We won't fine-tune CLIP
for param in model.parameters():
    param.requires_grad = False

# Get the image and text encoders
image_encoder = model.visual
text_encoder = model.encode_text  # Used in inference mode only

Let's prepare the train-test-splits.


In [ ]:
# get the three datasets
train_set, val_set, test_set = get_data(transform=preprocess)

# split classes into base and novel
base_classes, novel_classes = base_novel_categories(train_set)

# split the three datasets
train_base, _ = split_data(train_set, base_classes)
val_base, _ = split_data(val_set, base_classes)
test_base, test_novel = split_data(test_set, base_classes)

Now we will start implementing MoCoOp.


In [ ]:
# MoCoOp: Mixture of Prompt Learning

# utils
def l2norm(x, dim=-1, eps=1e-8):
    return x / (x.norm(dim=dim, keepdim=True) + eps)

# 8 groups of hard prompts for Flowers102
HARD_GROUPS = [
    ["a photo of a {}, a type of flower.", "a photo of the {}, a type of flower. "],    # flowers
    ["a photo of a {}.", "a photo of the {}."],                                     # generic
    ["a close-up photo of a {}.", "a macro photo of a {}."],                        # proximity
    ["a cropped photo of a {}.", "a cropped photo of the {}."],                     # crops
]

# function to encode text using CLIP's tokenizer and text encoder
@torch.no_grad()
def encode_texts(clip_model, texts):
    device = next(clip_model.parameters()).device # ensure device consistency
    tokenized = clip.tokenize(texts).to(device)  # [B, 77] - B is batch size, 77 is CLIP's max context length
    features = clip_model.encode_text(tokenized).float()  # [B, 512] - 512 is CLIP's text embedding dim
    return l2norm(features)

@torch.no_grad()
def build_hard_features(clip_model, class_names, hard_groups):
    """
    Assuming G groups, C classes and embedding dim D.
    Takes:
    Set of G groups of hard prompts.
    Returns:
    hard_group_avg: [G, D], averaged over group and classes. Will be used for router supervision.
    hard_group_class: [G, C, D] only averaged over group. Will be used for text-level supervision.
    """
    group_avg = []
    group_class = []
    for group in hard_groups:
        per_class = []
        for class_name in class_names:
            texts = [prompt.format(class_name) for prompt in group] # insert class name at placeholder {} in each prompt
            features = encode_texts(clip_model, texts).mean(0,keepdim=True)  # [1, D] - average across the 2 prompt embeddings in the group
            per_class.append(features)
        per_class = torch.cat(per_class, dim=0) # [C, D]
        per_class = l2norm(per_class)
        group_class.append(per_class)
        group_avg.append(per_class.mean(0, keepdim=True))
    hard_group_avg = l2norm(torch.cat(group_avg, dim=0)) # [G, D]
    hard_group_class = l2norm(torch.stack(group_class, dim=0)) # [G, C, D]
    return hard_group_avg, hard_group_class

# router: takes image features and outputs expert logits
class Router(nn.Module):
    def __init__(self, input_dim, G, hidden_dim=0, bias=True):
        super().__init__()
        if hidden_dim > 0:
            self.net = nn.Sequential(
                nn.Linear(input_dim, hidden_dim, bias=bias),
                nn.ReLU(inplace=True),
                nn.Dropout(p=0.5),
                nn.Linear(hidden_dim, G, bias=bias)
            )
        else:
            self.net = nn.Linear(input_dim, G, bias=bias)

    def forward(self, x):
        return self.net(x)      # logits: [B, G] where B is batch size

# text encoder: we cannot use CLIP's standard text encoder because we learn already embedded prompts, same as in CoOp.
class TextEncoder(nn.Module):
    def __init__(self, clip_model, token_prefix, token_suffix, tokenized_templates):
        super().__init__()
        self.clip_model = clip_model
        # freeze CLIP
        for p in self.clip_model.parameters():
            p.requires_grad = False
        self.token_prefix = token_prefix              # [C, 1, D] where C is the number of classes and D is the text embedding dim (512 for CLIP)
        self.token_suffix = token_suffix              # [C, S, D] where S is the number of tokens in the suffices
        self.tokenized_templates = tokenized_templates


    def forward(self, ctx_g):
        C = self.token_prefix.size(0)   # num of classes
        ctx = ctx_g.unsqueeze(0).expand(C, -1, -1)  # [C, n_ctx, D]
        x = torch.cat([self.token_prefix, ctx, self.token_suffix], dim=1)   # [C, seq_len, D]
        x = x + self.clip_model.positional_embedding[:x.size(1)]    # add positional embedding
        x = x.permute(1,0,2)  # [seq_len, C, D]
        x = self.clip_model.transformer(x)    # transformer
        x = x.permute(1,0,2)  # [C, seq_len, D]
        x = self.clip_model.ln_final(x) # layer normalization
        eot_idx = self.tokenized_templates.argmax(dim=-1)          # take index of EOT (highest in CLIP)
        x = x[torch.arange(C, device=x.device), eot_idx]           # extract hidden state at EOT token for each class.
        x = x @ self.clip_model.text_projection                     # text projection: matrix multiplication
        return l2norm(x)



# prompt experts
class PromptExperts(nn.Module):
    def __init__(self, clip_model, all_class_names, active_class_indices=None, n_ctx=16, hard_groups=None,
                top_k=2, tau=0.07, lambda_router=1.0, lambda_text=5.0,
                router_hidden=0):
        super().__init__()
        assert hard_groups is not None and len(hard_groups) > 0
        self.all_class_names = all_class_names
        self.all_C = len(all_class_names) # total number of classes (base + novel)
        
        # Active classes for classification (subset of all classes)
        if active_class_indices is None:
            active_class_indices = list(range(self.all_C))
        self.active_class_indices = active_class_indices
        self.active_C = len(active_class_indices) # number of active classes
        self.active_class_names = [all_class_names[i] for i in active_class_indices]
        
        self.n_ctx = n_ctx
        self.clip_model = clip_model
        # freeze CLIP
        for p in self.clip_model.parameters():
            p.requires_grad = False
        self.device = next(clip_model.parameters()).device
        self.dtype = next(clip_model.parameters()).dtype
        self.G = len(hard_groups) # number of experts
        self.top_k = top_k # number of active experts
        self.tau = tau
        self.lambda_router = lambda_router
        self.lambda_text = lambda_text

        # soft contexts
        D = clip_model.ln_final.weight.shape[0]  # CLIP's text embedding dimension, i.e. 512
        self.ctxs = nn.Parameter(torch.empty(self.G, n_ctx, D, dtype=self.dtype)) # [G, n_ctx, D] 

        # templates with placeholders - USE ALL CLASSES for text encoder
        prompt_prefix = " ".join(["X"] * n_ctx)
        template_prompts = [f"{prompt_prefix} {name}." for name in all_class_names]
        tokenized_template_prompts = torch.cat([clip.tokenize(p) for p in template_prompts]).to(self.device) # [all_C, 77]
        with torch.no_grad():
            embedded_template_prompts = clip_model.token_embedding(tokenized_template_prompts).type(self.dtype) # [all_C, 77, D]

        # register embeddings of prefices and suffices, and attention masks
        self.register_buffer("token_prefix", embedded_template_prompts[:, :1, :]) # embedding of SOT (start-of-text token): [all_C, 1, D]
        self.register_buffer("token_suffix", embedded_template_prompts[:, 1+n_ctx:, :]) # embedding of suffix (everything after context + EOT + padding): [all_C, suffix_len, D]
        self.tokenized_template_prompts = tokenized_template_prompts # will be useful later


        # hard-template initialization: per expert g, take the FIRST template of its group.
        # Split template at class placeholder, tokenize left/right parts separately,
        # concatenate embeddings (excluding class tokens), truncate to n_ctx or pad with noise.
        self.init_from_hard_templates(hard_groups)

        # text encoder
        self.text_encoder = TextEncoder(clip_model, self.token_prefix, self.token_suffix,
                                        self.tokenized_template_prompts)

        # router
        self.router = Router(input_dim=self.clip_model.visual.output_dim, G=self.G, hidden_dim=router_hidden)

        # hard-feature supervision buffers - USE ALL CLASSES for text supervision
        self.register_buffer("hard_group_avg", torch.empty(self.G, D))                 # [G, D]
        self.register_buffer("hard_group_class", torch.empty(self.G, self.all_C, D))  # [G, all_C, D]
        
        # hard-feature supervision buffers for ACTIVE CLASSES ONLY - for router supervision
        self.register_buffer("hard_group_avg_active", torch.empty(self.G, D))          # [G, D]

        # reuse CLIP scale
        self.logit_scale = clip_model.logit_scale

    def set_active_classes(self, active_class_indices):
        """Update which classes are active for classification"""
        self.active_class_indices = active_class_indices
        self.active_C = len(active_class_indices)
        self.active_class_names = [self.all_class_names[i] for i in active_class_indices]

    @torch.no_grad()
    def set_hard_features(self, hard_group_avg_all, hard_group_class_all):
        """Set hard features for ALL classes (used for text-level supervision)"""
        self.hard_group_avg.copy_(l2norm(hard_group_avg_all))
        self.hard_group_class.copy_(l2norm(hard_group_class_all))
        
        # Also compute hard features for active classes only (for router supervision)
        hard_group_avg_active = []
        for g in range(self.G):
            # Average over active classes only
            active_features = hard_group_class_all[g][self.active_class_indices] # [active_C, D]
            hard_group_avg_active.append(active_features.mean(0, keepdim=True))
        hard_group_avg_active = l2norm(torch.cat(hard_group_avg_active, dim=0))
        self.hard_group_avg_active.copy_(hard_group_avg_active)

    @torch.no_grad()
    def init_from_hard_templates(self, hard_groups):
        """
        Initializes the context parameters for each expert group using the first template in each hard prompt group.
        Args:
            hard_groups (list): List of hard prompt groups, each containing template strings with '{}' as a placeholder.
        Modifies:
            self.ctxs: Sets the context parameters for each group based on tokenized template embeddings.
        """
        token_embedding = self.clip_model.token_embedding
        n_ctx = self.ctxs.shape[1]
        D = self.ctxs.shape[2]

        def tok_no_class(text: str):
            toks = clip.tokenize([text]).to(self.device)[0]                 # [77]
            emb  = token_embedding(toks.unsqueeze(0)).type(self.dtype)[0]   # [77,D]
            # valid span = (after SOT) .. (before EOT)
            non_zero = (toks != 0).nonzero(as_tuple=False).flatten() # take indices of non zero tokens (Token ID 0 is padding token)
            if len(non_zero) == 0:
                return emb[:0]
            length = int(non_zero[-1].item()) + 1 # includes EOT
            return emb[1:length-1] # drop SOT and EOT -> [L,D]

        for g, group in enumerate(hard_groups):
            template = group[0]
            if "{}" not in template:
                raise ValueError(f"Template must contain '{{}}': {template}")

            left, right = template.split("{}", 1)
            left_emb  = tok_no_class(left)      # tokens before class
            right_emb = tok_no_class(right)     # tokens after class
            cat = torch.cat([left_emb, right_emb], dim=0)                   # [L',D], no class tokens

            # if ctx is longer than n_ctx -> truncate
            # if ctx is shorter -> pad with small random noise
            if cat.shape[0] >= n_ctx:
                ctx = cat[:n_ctx]
            else:
                pad = torch.randn(n_ctx - cat.shape[0], D, device=self.device, dtype=self.dtype) * 0.01
                ctx = torch.cat([cat, pad], dim=0)

            self.ctxs[g] = ctx

    def encode_all_experts(self):
        feats = [self.text_encoder(self.ctxs[g]) for g in range(self.G)] # list of [all_C, 512] where all_C is the total number of classes
        return torch.stack(feats, dim=0) # [G, all_C, 512]

    def forward(self, images, return_aux=True):
        # image features from frozen CLIP
        with torch.no_grad():
            img = self.clip_model.encode_image(images).float() # [B, D] where B is batch size and D is embedding dimension for images. i.e. 512
            img = l2norm(img)

        # router
        gate_logits = self.router(img) # [B, G]
        gate_probs = gate_logits.softmax(dim=-1) # [B, G]

        #text features for all experts and ALL classes
        txt_all = self.encode_all_experts() # [G, all_C, 512]
        
        # Extract features for active classes only for classification
        txt_active = txt_all[:, self.active_class_indices, :] # [G, active_C, 512]

        # top-k mixture per sample
        B = images.size(0)
        active_C = self.active_C
        D = txt_all.size(-1)
        K = self.top_k
        topk_probs, topk_idx = gate_probs.topk(K, dim=-1) # [B, K], [B, K]
        topk_probs = topk_probs / topk_probs.sum(-1, keepdim=True) # normalize top-k probabilities so they sum to 1

        # Get [B, K, active_C, D] text embeddings for top-K experts per sample (ACTIVE CLASSES ONLY)
        txt_topk = txt_active[topk_idx]  # topk_idx: [B, K] → txt_topk: [B, K, active_C, D]

        # Reshape weights to match: [B, K, 1, 1]
        weights = topk_probs.view(B, K, 1, 1)

        # Weighted sum over K experts → [B, active_C, D]
        mixed = (weights * txt_topk).sum(dim=1)

        # Normalize across D
        mixed = l2norm(mixed, dim=-1)

        # logits for ACTIVE CLASSES ONLY
        logit_scale = self.logit_scale.exp()
        logits = logit_scale * torch.einsum("bd,bcd->bc", img, mixed) # [B, active_C] cosine similarity between image and text features for each active class

        if not return_aux:
            return logits

        # regularizers: router KL to hard targets, and text-level supervision
        aux = {
            "gate_logits": gate_logits,
            "gate_prob": gate_probs,
            "topk_idx": topk_idx,
            "topk_prob": topk_probs,
        }

        # router KL - use ACTIVE CLASSES ONLY for router supervision
        with torch.no_grad():
            w_hard = (img @ l2norm(self.hard_group_avg_active).T).softmax(dim=-1) # [B, G]
        eps = 1e-8
        loss_router = -(w_hard * gate_probs.clamp_min(eps).log()).sum(dim=-1).mean() # KL divergence in disguise

        # text-level supervision - use ALL CLASSES for text supervision
        txt_soft_all = txt_all # [G, all_C, 512]
        with torch.no_grad():
            hard_norm = l2norm(self.hard_group_class) # [G, all_C, D]
        txt_soft_cmp = txt_soft_all
        losses_text = []
        for g in range(self.G):
            logits_g = (txt_soft_cmp[g] @ hard_norm[g].T) / self.tau # [all_C, all_C]
            target = torch.arange(self.all_C, device=images.device)
            losses_text.append(F.cross_entropy(logits_g, target))
        loss_text = torch.stack(losses_text).mean()

        aux["loss_router"] = loss_router
        aux["loss_text"] = loss_text
        aux["loss_total_reg"] = self.lambda_router * loss_router + self.lambda_text * loss_text
        return logits, aux

# CLIP wrapper
class MoCoOpCLIP(nn.Module):
    def __init__(self, clip_model, all_class_names, active_class_indices=None, hard_groups=None,
                n_ctx=16,
                top_k=2,
                tau=0.07,
                lambda_router=1.0,
                lambda_text=5.0,
                router_hidden=0):
        print(f"MoCoOp instance created | n_ctx={n_ctx}, lambda_text={lambda_text}, router_hidden={router_hidden}")
        super().__init__()
        # freeze CLIP
        for p in clip_model.parameters():
            p.requires_grad = False
        self.clip = clip_model

        # prompt experts
        self.prompt = PromptExperts(
            clip_model=self.clip,
            all_class_names=all_class_names,
            active_class_indices=active_class_indices,
            n_ctx=n_ctx,
            hard_groups=hard_groups,
            top_k=top_k,
            tau=tau,
            lambda_router=lambda_router,
            lambda_text=lambda_text,
            router_hidden=router_hidden
        )

    def set_active_classes(self, active_class_indices):
        """Update which classes are active for classification"""
        self.prompt.set_active_classes(active_class_indices)

    @torch.no_grad()
    def prime_hard(self, hard_group_avg, hard_group_class):
        self.prompt.set_hard_features(hard_group_avg, hard_group_class)

    def forward(self, images, targets=None):
        logits, aux = self.prompt(images, return_aux=True)
        if targets is None:
            return logits, aux
        loss_cls = F.cross_entropy(logits, targets)
        loss = loss_cls + aux.get("loss_total_reg", 0.0)
        aux["loss_cls"] = loss_cls
        aux["loss"] = loss
        return logits, aux

    def trainable_parameters(self):
        return [p for p in self.prompt.parameters() if p.requires_grad]

    def save(self, path):
        torch.save({"prompt_state": self.prompt.state_dict()}, path)

    def load(self, path, strict=True, map_location="cpu"):
        ckpt = torch.load(path, map_location=map_location)
        self.prompt.load_state_dict(ckpt["prompt_state"], strict=strict)

Training and eval


In [ ]:
# Remap base classes
train_base_remapped = create_remapped_dataset(train_set, base_classes)
val_base_remapped   = create_remapped_dataset(val_set, base_classes)
test_base_remapped  = create_remapped_dataset(test_set, base_classes)

train_base_dataset = RemappedDataset(train_base_remapped)
val_base_dataset   = RemappedDataset(val_base_remapped)
test_base_dataset  = RemappedDataset(test_base_remapped)

batch_size = 8
train_loader = torch.utils.data.DataLoader(train_base_dataset, batch_size=batch_size, shuffle=True,  num_workers=2)
val_loader   = torch.utils.data.DataLoader(val_base_dataset,   batch_size=batch_size, shuffle=False, num_workers=2)
test_loader  = torch.utils.data.DataLoader(test_base_dataset,  batch_size=batch_size, shuffle=False, num_workers=2)

In [ ]:
# Build MoCoOp with ALL classes but only base classes active for training
all_class_names = CLASS_NAMES
mocoop = MoCoOpCLIP(model, all_class_names=all_class_names, active_class_indices=base_classes, hard_groups=HARD_GROUPS).to(device)

# Prime hard features with ALL classes (for text-level supervision)
with torch.no_grad():
    hard_avg_all, hard_cls_all = build_hard_features(model, all_class_names, HARD_GROUPS)
mocoop.prime_hard(hard_avg_all, hard_cls_all)

# Optimizer over trainables only: ctxs + router
params = [p for p in mocoop.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.002, momentum=0.9)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200, eta_min=1e-4)

num_epochs = 5
best_val = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_acc': []}

In [ ]:
def evaluate_mocoop(model, loader, device):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits, _ = model(images)                # targets=None -> returns logits, aux
            pred = logits.argmax(dim=1)
            correct += (pred == labels).sum().item()
            total   += labels.numel()
    return correct / max(total, 1)

print(f"Trainable params: {sum(p.numel() for p in params):,}")

mocoop.train()

In [ ]:
for epoch in range(num_epochs):
    mocoop.train()
    running_loss = 0.0
    correct = total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        images = images.to(torch.float32) # Explicitly cast images to float32

        optimizer.zero_grad()
        logits, aux = mocoop(images, labels)        # returns logits and aux with aux['loss']
        loss = aux['loss']                           # CE + router/text regularizers (already summed inside)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total   += images.size(0)

    scheduler.step()
    train_loss = running_loss / total
    train_acc  = correct / total
    val_acc    = evaluate_mocoop(mocoop, val_loader, device)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    if val_acc > best_val:
        best_val = val_acc
        # Save only prompt state (your class exposes .save/.load)
        mocoop.save('best_mocoop_prompt.pth')
    print(f"Epoch {epoch+1}/{num_epochs} | loss {train_loss:.4f} | acc {train_acc:.4f} | val {val_acc:.4f}")

print(f"Best val: {best_val:.4f}")

# Load best prompt and test on base
mocoop.load('best_mocoop_prompt.pth', strict=False, map_location=device)   # loads into mocoop.prompt
base_test_acc = evaluate_mocoop(mocoop, test_loader, device)
print(f"Base test acc: {base_test_acc:.4f}")

# ===== Novel class evaluation =====
# Switch to novel classes (reuse same model, just change active classes)
mocoop.set_active_classes(novel_classes)

# Prime hard features again - still use ALL classes for text supervision, but router uses novel classes
with torch.no_grad():
    hard_avg_all, hard_cls_all = build_hard_features(model, all_class_names, HARD_GROUPS)
mocoop.prime_hard(hard_avg_all, hard_cls_all)

# Novel dataset and loader
test_novel_remapped = create_remapped_dataset(test_set, novel_classes)
test_novel_dataset  = RemappedDataset(test_novel_remapped)
test_novel_loader   = torch.utils.data.DataLoader(test_novel_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

novel_test_acc = evaluate_mocoop(mocoop, test_novel_loader, device)
print(f"Novel test acc: {novel_test_acc:.4f}")
print(f"Harmonic mean: {(2 / (1/base_test_acc + 1/novel_test_acc)):.4f}")